In [5]:
# ==========================================
# Experiment 5: Text Summarization + Question Answering
# Using Hugging Face Transformer Models
# Google Colab Compatible Version
# ==========================================

!pip -q install transformers torch sentencepiece

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForQuestionAnswering
)
import torch

print("Loading models...")


# ------------------------------------------
# 1. TEXT SUMMARIZATION
# ------------------------------------------

summary_model_name = "facebook/bart-large-cnn"

summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)

summary_model = AutoModelForSeq2SeqLM.from_pretrained(
    summary_model_name
)

print("Summarization model loaded!")


article = """
Generative AI refers to a class of artificial intelligence models capable of
producing new content such as text, images, audio, and video. Large Language
Models (LLMs) such as GPT and LLaMA are trained on massive text corpora and
can perform a wide range of natural language tasks including translation,
summarization, and question answering. These models are increasingly being
deployed in industry applications ranging from customer support to software
development, transforming how humans interact with machines.
"""


# Tokenize article
inputs = summary_tokenizer(
    article,
    return_tensors="pt",
    max_length=1024,
    truncation=True
)


# Generate summary
summary_ids = summary_model.generate(
    inputs["input_ids"],
    max_length=45,
    min_length=20,
    do_sample=False
)


summary = summary_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)


print("\n==============================")
print("TEXT SUMMARY")
print("==============================")
print(summary)



# ------------------------------------------
# 2. QUESTION ANSWERING
# ------------------------------------------

qa_model_name = "distilbert-base-cased-distilled-squad"


qa_tokenizer = AutoTokenizer.from_pretrained(
    qa_model_name
)

qa_model = AutoModelForQuestionAnswering.from_pretrained(
    qa_model_name
)


print("\nQuestion Answering model loaded!")


context = article

question = "What are Large Language Models trained on?"


# Tokenize question and context
qa_inputs = qa_tokenizer(
    question,
    context,
    return_tensors="pt"
)


with torch.no_grad():
    outputs = qa_model(**qa_inputs)


start_scores = outputs.start_logits
end_scores = outputs.end_logits


start_index = torch.argmax(start_scores)
end_index = torch.argmax(end_scores)


answer_tokens = qa_inputs["input_ids"][0][start_index:end_index+1]

answer = qa_tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)


confidence = (
    torch.softmax(start_scores, dim=1).max() *
    torch.softmax(end_scores, dim=1).max()
)


print("\n==============================")
print("QUESTION ANSWERING")
print("==============================")

print("Question:", question)
print("Answer:", answer)
print("Confidence:", round(float(confidence), 3))

Loading models...


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

...

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]


Question Answering model loaded!

QUESTION ANSWERING
Question: What are Large Language Models trained on?
Answer: massive text corpora
Confidence: 0.892
